In [17]:
# Setup imports and Kaggle API
import os
import sys
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path to use our data_loader
project_root = Path().absolute().parent.parent
sys.path.append(str(project_root))

# Setup Kaggle API authentication
os.environ['KAGGLE_CONFIG_DIR'] = str(project_root / '.kaggle')

# Import Kaggle API
from kaggle.api.kaggle_api_extended import KaggleApi

# Initialize Kaggle API
api = KaggleApi()
api.authenticate()

print("Kaggle API authenticated successfully!")


Kaggle API authenticated successfully!


In [18]:
# Download dataset using Kaggle API
dataset_name = 'ellipticco/elliptic-data-set'
download_path = project_root / 'data'
data_path = download_path / 'elliptic_bitcoin_dataset'

# Create directory if it doesn't exist
download_path.mkdir(parents=True, exist_ok=True)

# Check if dataset already exists
csv_files = list(data_path.glob('*.csv'))
if not csv_files:
    print(f"Downloading dataset to {download_path}...")
    
    api.dataset_download_files(
        dataset_name, 
        path=str(download_path),
        unzip=True,
        quiet=False
    )
    print("Download complete!")
    
# List downloaded files
print("\nDataset files:")
for file in sorted(data_path.glob('*.csv')):
    print(f"  - {file.name} ({file.stat().st_size / 1024 / 1024:.1f} MB)")



Dataset files:
  - elliptic_txs_classes.csv (3.2 MB)
  - elliptic_txs_edgelist.csv (4.3 MB)
  - elliptic_txs_features.csv (657.7 MB)


In [19]:
# Load data using our data loader
from src.data_loader import EllipticDataLoader

# Initialize data loader
loader = EllipticDataLoader(data_dir=data_path)



# Load the dataset
elliptic_txs_features, elliptic_txs_classes, elliptic_txs_edgelist = loader.load_data()

# The data loader already handles column naming for features
# Let's verify the shapes
print(f"""Shapes
{4*' '}Features : {elliptic_txs_features.shape[0]:8,} (rows)  {elliptic_txs_features.shape[1]:4,} (cols)
{4*' '}Classes  : {elliptic_txs_classes.shape[0]:8,} (rows)  {elliptic_txs_classes.shape[1]:4,} (cols)
{4*' '}Edgelist : {elliptic_txs_edgelist.shape[0]:8,} (rows)  {elliptic_txs_edgelist.shape[1]:4,} (cols)
""")

Loaded features: (203769, 167)
Loaded classes: (203769, 2)
Loaded edges: (234355, 2)
Shapes
    Features :  203,769 (rows)   167 (cols)
    Classes  :  203,769 (rows)     2 (cols)
    Edgelist :  234,355 (rows)     2 (cols)



## Understanding the Three Dataframes

The Elliptic dataset consists of three interconnected dataframes that together form a temporal graph of Bitcoin transactions:


In [22]:
# 1. FEATURES DATAFRAME (elliptic_txs_features)
# This is the main dataframe containing transaction features

print("=" * 60)
print("1. FEATURES DATAFRAME")
print("=" * 60)

# Display basic info
print(f"Shape: {elliptic_txs_features.shape}")
print(f"Columns: {elliptic_txs_features.shape[1]}")
print(f"TrueFeatures: {elliptic_txs_features.shape[1] - 1}")  # Exclude txId
print(f"Transactions: {elliptic_txs_features.shape[0]:,}")

# Show column structure
print("\nColumn Structure:")
print(f"- Column 0: Transaction ID (txId)")
print(f"- Columns 1-94: Local features (transaction-specific)")
print(f"  - Time step (1-49, ~2 week intervals)")
print(f"  - Transaction metrics: inputs/outputs, fees, volumes")
print(f"  - Aggregated statistics from inputs/outputs")
print(f"- Columns 95-166: Neighborhood features (1-hop aggregations)")
print(f"  - Statistical measures (min, max, std, correlation)")
print(f"  - Computed from connected transactions")

# Display first few rows
print("\nFirst 5 transactions:")
print(elliptic_txs_features.head())

# Check for any missing values
print(f"\nMissing values: {elliptic_txs_features.isnull().sum().sum()}")


1. FEATURES DATAFRAME
Shape: (203769, 167)
Columns: 167
TrueFeatures: 166
Transactions: 203,769

Column Structure:
- Column 0: Transaction ID (txId)
- Columns 1-94: Local features (transaction-specific)
  - Time step (1-49, ~2 week intervals)
  - Transaction metrics: inputs/outputs, fees, volumes
  - Aggregated statistics from inputs/outputs
- Columns 95-166: Neighborhood features (1-hop aggregations)
  - Statistical measures (min, max, std, correlation)
  - Computed from connected transactions

First 5 transactions:
         0    1         2         3         4          5         6    \
0  230425980    1 -0.171469 -0.184668 -1.201369  -0.121970 -0.043875   
1    5530458    1 -0.171484 -0.184668 -1.201369  -0.121970 -0.043875   
2  232022460    1 -0.172107 -0.184668 -1.201369  -0.121970 -0.043875   
3  232438397    1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
4  230460314    1  1.011523 -0.081127 -1.201369   1.153668  0.333276   

        7          8         9    ...       

In [28]:
# 2. CLASSES DATAFRAME (elliptic_txs_classes)
print("=" * 60)
print("2. CLASSES DATAFRAME")
print("=" * 60)


# Calculate percentages
print("\nClass Percentages:")
for class_name, count in class_counts.items():
    percentage = (count / len(elliptic_txs_classes)) * 100
    print(f"- {class_name}: {count:,} ({percentage:.2f}%)")

# Show sample of labeled data
print("\nSample of labeled transactions:")
print(elliptic_txs_classes.head(10))

# Check which transactions have labels
labeled_txs = elliptic_txs_classes[elliptic_txs_classes['class'].isin(['1', '2'])]
print(f"\nLabeled transactions: {len(labeled_txs):,} out of {len(elliptic_txs_classes):,}")
print(f"Unlabeled (unknown): {elliptic_txs_classes['class'].value_counts().get('unknown', 0):,}")

# Note about class meanings
print("\nClass Labels:")
print("- '1': Illicit (illegal) transaction")
print("- '2': Licit (legal) transaction") 
print("- 'unknown': Unlabeled transaction")


2. CLASSES DATAFRAME

Class Percentages:
- unknown: 157,205 (77.15%)
- 2: 42,019 (20.62%)
- 1: 4,545 (2.23%)

Sample of labeled transactions:
        txId    class
0  230425980  unknown
1    5530458  unknown
2  232022460  unknown
3  232438397        2
4  230460314  unknown
5  230459870  unknown
6  230333930  unknown
7  230595899  unknown
8  232013274  unknown
9  232029206        2

Labeled transactions: 46,564 out of 203,769
Unlabeled (unknown): 157,205

Class Labels:
- '1': Illicit (illegal) transaction
- '2': Licit (legal) transaction
- 'unknown': Unlabeled transaction


In [30]:
# 3. EDGELIST DATAFRAME (elliptic_txs_edgelist)
# This dataframe defines the graph structure - connections between transactions

print("=" * 60)
print("3. EDGELIST DATAFRAME")
print("=" * 60)

# Display basic info
print(f"Shape: {elliptic_txs_edgelist.shape}")
print(f"Columns: {list(elliptic_txs_edgelist.columns)}")
print(f"Total edges (connections): {len(elliptic_txs_edgelist):,}")

# Show sample edges
print("\nSample edges (first 10):")
print(elliptic_txs_edgelist.head(10))

# Analyze graph properties
unique_nodes = pd.concat([
    elliptic_txs_edgelist['txId1'], 
    elliptic_txs_edgelist['txId2']
]).nunique()

print(f"\nGraph Statistics:")
print(f"- Unique nodes in edge list: {unique_nodes:,}")
print(f"- Total edges: {len(elliptic_txs_edgelist):,}")
print(f"- Average degree: {(2 * len(elliptic_txs_edgelist)) / unique_nodes:.2f}")

# Check if graph is directed or undirected
# If undirected, each edge should appear only once
reverse_edges = pd.merge(
    elliptic_txs_edgelist,
    elliptic_txs_edgelist.rename(columns={'txId1': 'txId2', 'txId2': 'txId1'}),
    on=['txId1', 'txId2'],
    how='inner'
)
print(f"\nGraph type: {'Undirected' if len(reverse_edges) == 0 else 'Directed'}")

# Important note about temporal structure
print("\nIMPORTANT: Graph Temporal Structure")
print("- Edges only connect transactions within the same time step")
print("- Transactions in an edge occurred within ~3 hours of each other")
print("- No edges exist between different time steps (1-49)")
print("- This creates 49 separate connected components")


3. EDGELIST DATAFRAME
Shape: (234355, 2)
Columns: ['txId1', 'txId2']
Total edges (connections): 234,355

Sample edges (first 10):
       txId1      txId2
0  230425980    5530458
1  232022460  232438397
2  230460314  230459870
3  230333930  230595899
4  232013274  232029206
5  232344069   27553029
6   36411953  230405052
7   34194980    5529846
8    3881097  232457116
9  230409257   32877982

Graph Statistics:
- Unique nodes in edge list: 203,769
- Total edges: 234,355
- Average degree: 2.30

Graph type: Undirected

IMPORTANT: Graph Temporal Structure
- Edges only connect transactions within the same time step
- Transactions in an edge occurred within ~3 hours of each other
- No edges exist between different time steps (1-49)
- This creates 49 separate connected components


In [29]:
print("=" * 60)
print("77% OF TRANSACTIONS ARE UNLABELED")
print("=" * 60)

# Quick breakdown
total_tx = len(elliptic_txs_classes)
class_dist = elliptic_txs_classes['class'].value_counts()

print(f"\nTotal transactions: {total_tx:,}")
print(f"- Illicit (class 1): {class_dist.get('1', 0):,} ({class_dist.get('1', 0)/total_tx*100:.1f}%)")
print(f"- Licit (class 2): {class_dist.get('2', 0):,} ({class_dist.get('2', 0)/total_tx*100:.1f}%)")
print(f"- Unknown: {class_dist.get('unknown', 0):,} ({class_dist.get('unknown', 0)/total_tx*100:.1f}%)")


77% OF TRANSACTIONS ARE UNLABELED

Total transactions: 203,769
- Illicit (class 1): 4,545 (2.2%)
- Licit (class 2): 42,019 (20.6%)
- Unknown: 157,205 (77.1%)


In [34]:
# GRAPH CONNECTIVITY ANALYSIS
print("=" * 80)
print("CHECKING IF THE GRAPH IS FULLY CONNECTED")
print("=" * 80)

import networkx as nx

# Create a NetworkX graph from the edgelist
print("Creating NetworkX graph from edgelist...")
G = nx.from_pandas_edgelist(elliptic_txs_edgelist, 
                           source='txId1', 
                           target='txId2', 
                           create_using=nx.DiGraph())

print(f"Graph created with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges")

# Check if graph is connected (for directed graphs, we check weak connectivity)
is_weakly_connected = nx.is_weakly_connected(G)
print(f"\nIs weakly connected: {is_weakly_connected}")

if not is_weakly_connected:
    # Find connected components
    weak_components = list(nx.weakly_connected_components(G))
    print(f"Number of weakly connected components: {len(weak_components)}")
    
    # Show sizes of largest components
    component_sizes = [len(comp) for comp in weak_components]
    component_sizes.sort(reverse=True)
    
    print(f"\nLargest connected components:")
    for i, size in enumerate(component_sizes[:10]):
        percentage = (size / G.number_of_nodes()) * 100
        print(f"  Component {i+1}: {size:,} nodes ({percentage:.1f}%)")
    
    # Check if this correlates with time steps
    print(f"\nTotal nodes in all components: {sum(component_sizes):,}")
    print(f"Nodes in features dataset: {len(elliptic_txs_features):,}")
    
    # Analyze if components align with time steps
    print("\nChecking if components align with time steps...")
    
    # Get time step for each node
    # Column 0 contains txId, Column 1 contains the time step
    tx_timesteps = elliptic_txs_features.set_index(elliptic_txs_features.columns[0])[elliptic_txs_features.columns[1]]
    
    # Check first few components
    for i, component in enumerate(weak_components[:5]):
        if len(component) > 100:  # Only check larger components
            timesteps_in_comp = []
            for node in component:
                if node in tx_timesteps.index:
                    timesteps_in_comp.append(tx_timesteps[node])
            
            if timesteps_in_comp:
                unique_timesteps = set(timesteps_in_comp)
                print(f"  Component {i+1} ({len(component):,} nodes): Time steps {sorted(unique_timesteps)}")

else:
    print("The graph is fully connected!")

print("\nKEY INSIGHTS:")
print("- If graph has many components, it suggests temporal or structural isolation")
print("- Bitcoin transaction graphs are typically fragmented")
print("- Each component might represent a separate 'cluster' of related transactions")


CHECKING IF THE GRAPH IS FULLY CONNECTED
Creating NetworkX graph from edgelist...
Graph created with 203,769 nodes and 234,355 edges

Is weakly connected: False
Number of weakly connected components: 49

Largest connected components:
  Component 1: 7,880 nodes (3.9%)
  Component 2: 7,140 nodes (3.5%)
  Component 3: 6,803 nodes (3.3%)
  Component 4: 6,727 nodes (3.3%)
  Component 5: 6,621 nodes (3.2%)
  Component 6: 6,393 nodes (3.1%)
  Component 7: 6,048 nodes (3.0%)
  Component 8: 5,894 nodes (2.9%)
  Component 9: 5,693 nodes (2.8%)
  Component 10: 5,598 nodes (2.7%)

Total nodes in all components: 203,769
Nodes in features dataset: 203,769

Checking if components align with time steps...
  Component 1 (7,880 nodes): Time steps [np.int64(1)]
  Component 2 (4,544 nodes): Time steps [np.int64(2)]
  Component 3 (6,621 nodes): Time steps [np.int64(3)]
  Component 4 (5,693 nodes): Time steps [np.int64(4)]
  Component 5 (6,803 nodes): Time steps [np.int64(5)]

KEY INSIGHTS:
- If graph has m